In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score

In [21]:
raw_original_data_path = Path("../data/raw/netflix_customer_churn.csv")
to_predict_data_path = Path("../data/to_predict/netflix_customer_churn_new_1000_to_predict.csv")
processed_data_path = Path("../data/processed")
models_path = Path("../models")

In [22]:
df_to_predict = pd.read_csv(to_predict_data_path)
df = df_to_predict.copy()

In [23]:
df.drop(
    columns=[
        'customer_id',
        'subscription_type',
        'avg_watch_time_per_day',
        'gender',
        'region',
        'device',
        'favorite_genre',
        'age',
    ],
    inplace=True,
)

In [24]:
df_clean = df.copy()
df_clean.head()

,watch_hours,last_login_days,monthly_fee,payment_method,number_of_profiles
0,24.43,48,8.99,Crypto,3
1,8.66,10,17.99,PayPal,5
2,48.71,11,13.99,PayPal,3
3,82.07,44,17.99,Debit Card,3
4,15.12,40,13.99,Crypto,2


In [25]:
def add_features(df):
    df = df.copy()

    df["log_watch_hours"] = np.log1p(df["watch_hours"].clip(lower=0))
    df["inactive_30d"] = (df["last_login_days"] > 30).astype(int)
    df["profiles_4plus"] = (df["number_of_profiles"] >= 4).astype(int)
    df["risky_payment"] = (df["payment_method"].isin(["Crypto", "Gift Card"])).astype(int)

    return df

df_featured = add_features(df_clean)
df_featured.head()

,watch_hours,last_login_days,monthly_fee,payment_method,number_of_profiles,log_watch_hours,inactive_30d,profiles_4plus,risky_payment
0,24.43,48,8.99,Crypto,3,3.235930,1,0,1
1,8.66,10,17.99,PayPal,5,2.267994,0,1,0
2,48.71,11,13.99,PayPal,3,3.906206,0,0,0
3,82.07,44,17.99,Debit Card,3,4.419684,1,0,0
4,15.12,40,13.99,Crypto,2,2.780061,1,0,1


In [26]:
preprocessor = joblib.load(models_path / "preprocessor.joblib")

In [32]:
X_new_arr = preprocessor.transform(df_featured)
X_new_processed = pd.DataFrame(
    X_new_arr,
    columns=preprocessor.get_feature_names_out(),
    index=df_featured.index,
)

X_new_processed.head()

,num__watch_hours,num__last_login_days,num__monthly_fee,num__number_of_profiles,num__log_watch_hours,cat__payment_method_Crypto,cat__payment_method_Debit Card,cat__payment_method_Gift Card,cat__payment_method_PayPal,bin__inactive_30d,bin__profiles_4plus,bin__risky_payment
0,1.288556,0.600000,-0.555556,0.0,0.755527,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0.046090,-0.666667,0.444444,1.0,0.045805,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,3.201497,-0.633333,0.000000,0.0,1.246995,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,5.829821,0.466667,0.444444,0.0,1.623493,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.555052,0.333333,0.000000,-0.5,0.421269,1.0,0.0,0.0,0.0,1.0,0.0,1.0


In [28]:
models = {
    "Logistic Regression": joblib.load(models_path / "logistic_regression.joblib"),
    "Decision Tree (depth=3)": joblib.load(models_path / "decision_tree_3_depth.joblib"),
    "Decision Tree (depth=5)": joblib.load(models_path / "decision_tree_5_depth.joblib"),
}

In [31]:
results = []
for name, clf in models.items():
    y_pred = clf.predict(X_new_processed)
    y_prob = clf.predict_proba(X_new_processed)[:, 1]
    results.append({
        "Model": name,
        "Predicted Churn (1)": int((y_pred == 1).sum()),
        "Predicted Stay (0)": int((y_pred == 0).sum()),
        "Churn Rate (%)": round(y_pred.mean() * 100, 2),
        "Avg Churn Prob": round(y_prob.mean(), 4)
    })

comparison_df = pd.DataFrame(results).sort_values(by="Avg Churn Prob", ascending=False)
display(comparison_df)

,Model,Predicted Churn (1),Predicted Stay (0),Churn Rate (%),Avg Churn Prob
2,Decision Tree (depth=5),484,516,48.4,0.5129
0,Logistic Regression,527,473,52.7,0.5124
1,Decision Tree (depth=3),566,434,56.6,0.5078


In [36]:
best_model = models["Decision Tree (depth=5)"]
df_to_predict["churned"] = best_model.predict(X_new_processed)

output_path = to_predict_data_path.parent / "netflix_customer_churn_predicted.csv"
df_to_predict.to_csv(output_path, index=False)